# pysndlib basics

pysndlib is a python wrapper around sndlib, Bill Schottstaedt's CLM sound library. It follows the design of Common Lisp Music (CLM): you create **generators** (oscillators, envelopes, filters, delay lines -- made once, called once per sample), and render inside a **`Sound` context manager** that opens the output file, writes your samples, and closes the header when the block ends.

This notebook: a first sound, reading it back, and where to go next. It runs top to bottom (requires `pip install pysndlib numpy matplotlib`).

Docs: [installation](https://testcase.github.io/pysndlib/installation.html) | [basic usage](https://testcase.github.io/pysndlib/basic_usage.html) | [what's new in 0.8](https://testcase.github.io/pysndlib/whats_new.html) | [settings file](https://testcase.github.io/pysndlib/settings.html) | [CLM reference](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import pysndlib.clm as clm
import pysndlib.sndlib as sndlib
import numpy as np
import IPython.display as ipd
from matplotlib import pyplot as plt

SR = 44100

## The Sound context and your first render

Everything renders inside `clm.Sound(...)`. By default it writes `test.wav` (see the settings page for how to change that, and for all the `Sound` options). Inside the block you make generators and then compute one sample at a time -- the CLM idiom:

- `make_oscil(freq)` -- a sine oscillator
- `oscil(gen, fm)` -- read the next sample; the second argument is an FM input
- `make_env([...], length=...)` -- an envelope over a breakpoint list `[x0, y0, x1, y1, ...]`
- `outa(loc, value)` -- write a sample to channel A

Here a modulator at 231 Hz drives the carrier's phase, with an envelope fading the FM depth in -- a simple FM bell:

In [ ]:
with clm.Sound(play=False):              # writes default.file_name ('test.wav')
    osc = clm.make_oscil(300)            # carrier
    mod = clm.make_oscil(231)            # modulator
    ienv = clm.make_env([0.0, 0.0, 1.0, 1.0], length=44100)
    for i in range(44100):               # one second of samples
        clm.outa(i, clm.oscil(osc, clm.oscil(mod) * .01 * clm.env(ienv)))

# read the file back and listen (embed the samples -- plays anywhere)
data, sr = clm.file2ndarray('test.wav')
ipd.Audio(data=data[0], rate=int(sr))

## Reading back: data and metadata

`file2ndarray` returns the samples as a numpy array (shape `(chans, framples)`) plus the file's sampling rate. The `mus_sound_*` functions read header information without loading any data:

In [ ]:
print('samples  :', sndlib.mus_sound_samples('test.wav'))    # frames * chans
print('framples :', sndlib.mus_sound_framples('test.wav'))  # frames per channel
print('srate     :', sndlib.mus_sound_srate('test.wav'))
print('chans     :', sndlib.mus_sound_chans('test.wav'))
print('header    :', sndlib.mus_header_type_to_string(sndlib.mus_sound_header_type('test.wav')))
print('sample type:', sndlib.mus_sample_type_to_string(sndlib.mus_sound_sample_type('test.wav')))
print('duration :', sndlib.mus_sound_duration('test.wav'), 'seconds')

## The waveform

In [ ]:
plt.figure(figsize=(18, 2.5))
plt.plot(data[0], color='gray')
plt.xlabel('Time (samples)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## Where to go next

The rest of the series (see the repository's `notebooks/` directory):

| # | Notebook | Topics |
|---|----------|--------|
| 02 | [Generators and Envelopes](02_Generators_and_Envelopes.ipynb) | `Sound` anatomy, oscil/env/FM, waveform alternatives, noise |
| 03 | [Filters, Delays and DSP](03_Filters_Delay_DSP.ipynb) | one-pole, formant banks, delay lines, the `dsp` module |
| 04 | [Instruments](04_Instruments.ipynb) | writing instruments, the shipped collection, reverb -- [see also the instruments docs](https://testcase.github.io/pysndlib/instruments.html) |
| 05 | [Speed with Cython](05_Speed_with_Cython.ipynb) | `%%cython`, typed locals, benchmarks |
| 06-08 | [FM trilogy](06_FM_Theory.ipynb) | Bessel theory, classic instruments, advanced FM |
| 09 | [Waveforms](09_Waveforms.ipynb) | the generator zoo: naive classics, polynomial, wavetables |
| 10 | [Noise](10_Noise.ipynb) | rand/rand_interp, pink/brown noise, smoothing |
| 11 | [Delays and Physical Modeling](11_Delays_Physical.ipynb) | comb resonance, strings, reverb internals |
| 12 | [Granulate](12_Granulate.ipynb) | granular synthesis |
| 13 | [Phase Vocoder](13_Phase_Vocoder.ipynb) | pitch/time decoupling |
| 14-16 | Spectral tools, sound files, spatialization | 